In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance,VectorParams,PointStruct

In [2]:
load_dotenv()
client=OpenAI()

In [3]:
def get_embedding(text,model="text-embedding-3-small"):
    response=client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

Reterivel Function

In [4]:
quadrant_client=QdrantClient(url="http://localhost:6333/")

In [58]:
def reteriver_data(query, qdrant_client, k):
    query_embedding = get_embedding(query)

    result = qdrant_client.query_points(
        collection_name="Amazon_items_collection-00",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_rating = []

    for point in result.points:
        retrieved_context_ids.append(point.payload["parent_asin"])
        retrieved_context.append(point.payload["description"])
        retrieved_context_rating.append(point.payload["average_rating"])
        similarity_scores.append(point.score)

    return (
        retrieved_context,
        retrieved_context_ids,
        retrieved_context_rating,
        similarity_scores
    )

In [48]:
contexts = retriever_data(
    "what kind of charging cords do you offer",
    quadrant_client,
    5
)


In [53]:
contexts

(['5pack iPhone Charger Lightning Cables for Phone 13 13pro Max 12 Mini 11 Se 10 X Xr Xs 8 7 6 6s, Apple Certified Original Braided Cord Multi-Color Lightening Wire Fast Charging Cargador Chord-3 6 10 FtAPPLE MFI CERTIFIED - Made with original Apple MFI Chip, full support with iOS versions. High quality, pull-resistance, fast charging, and transmission.USB A TO LIGHTNING - USB-A to USB-Lightning port cable matches with USB A Wall charger. you can charge your iPhone, iPad, iPod. But not fit for android micro or type c port phones. 【Please make sure this is not a USB-C to LIGHTNING charging cord】UNIVERSAL COMPATIBILITY - Durable braided lightning cords are fully compatible with Apple iPhone 14 Pro Max, 14 Plus, 14 Pro, 14, 13 Pro Max, 13 Pro, 13, 13 mini, 12 Pro Max, 12 Pro, 12, 12 mini, SE, 11 Pro Max, 11 Pro, 11, XS Max, XS, XR, X, 8Plus, 8, 7 Plus, 7, SE, 6s Plus, 6s, 6Plus, 6; iPad 10.2 (2021), iPad 10.2 (2020), iPad 10.2 (2019), iPad 9.7 (2018), iPad 9.7 (2017); iPod nano, iPod touc

Format retrived Context Function

In [51]:
def process_context(context, ids, ratings, scores):
    formatted_context = ""

    for id, chunk, rating, score in zip(ids, context, ratings, scores):
        formatted_context += (
            f"- ID: {id}\n"
            f"  Description: {chunk}\n"
            f"  Rating: {rating}\n"
            f"  Similarity Score: {score:.4f}\n\n"
        )

    return formatted_context

In [52]:


formatted = process_context(contexts, ids, ratings, scores)
print(formatted)


- ID: B0BPLX388R
  Description: ['5pack iPhone Charger Lightning Cables for Phone 13 13pro Max 12 Mini 11 Se 10 X Xr Xs 8 7 6 6s, Apple Certified Original Braided Cord Multi-Color Lightening Wire Fast Charging Cargador Chord-3 6 10 FtAPPLE MFI CERTIFIED - Made with original Apple MFI Chip, full support with iOS versions. High quality, pull-resistance, fast charging, and transmission.USB A TO LIGHTNING - USB-A to USB-Lightning port cable matches with USB A Wall charger. you can charge your iPhone, iPad, iPod. But not fit for android micro or type c port phones. 【Please make sure this is not a USB-C to LIGHTNING charging cord】UNIVERSAL COMPATIBILITY - Durable braided lightning cords are fully compatible with Apple iPhone 14 Pro Max, 14 Plus, 14 Pro, 14, 13 Pro Max, 13 Pro, 13, 13 mini, 12 Pro Max, 12 Pro, 12, 12 mini, SE, 11 Pro Max, 11 Pro, 11, XS Max, XS, XR, X, 8Plus, 8, 7 Plus, 7, SE, 6s Plus, 6s, 6Plus, 6; iPad 10.2 (2021), iPad 10.2 (2020), iPad 10.2 (2019), iPad 9.7 (2018), iPad 9

Create Prompt Function

In [25]:
def build_prompt(processed_context, question):
    prompt = f"""
You are a helpful AI shopping assistant.

Use ONLY the information provided in the retrieved product context below to answer the user's question.

Rules:
- Do not make up information.
- If the answer is not present in the context, reply:
  "I couldn't find that information in the retrieved products."
- Keep your answer concise and helpful.
- Mention product IDs when relevant.

======================
Retrieved Context:
{processed_context}
======================

User Question:
{question}

Answer:
"""

    return prompt

In [34]:
prompt=build_prompt(formatted,"what type of cords can i get")

In [35]:
print(prompt)


You are a helpful AI shopping assistant.

Use ONLY the information provided in the retrieved product context below to answer the user's question.

Rules:
- Do not make up information.
- If the answer is not present in the context, reply:
  "I couldn't find that information in the retrieved products."
- Keep your answer concise and helpful.
- Mention product IDs when relevant.

Retrieved Context:
- ID: B0BPLX388R
  Description: 5pack iPhone Charger Lightning Cables for Phone 13 13pro Max 12 Mini 11 Se 10 X Xr Xs 8 7 6 6s, Apple Certified Original Braided Cord Multi-Color Lightening Wire Fast Charging Cargador Chord-3 6 10 FtAPPLE MFI CERTIFIED - Made with original Apple MFI Chip, full support with iOS versions. High quality, pull-resistance, fast charging, and transmission.USB A TO LIGHTNING - USB-A to USB-Lightning port cable matches with USB A Wall charger. you can charge your iPhone, iPad, iPod. But not fit for android micro or type c port phones. 【Please make sure this is not a USB

In [42]:

def generate_chat(prompt):
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": prompt
            },
            
        ],
        temperature=0.2,
        max_tokens=300
    )

    return response.choices[0].message.content

In [43]:
res=generate_chat(prompt)

In [44]:
print(res)

You can get Lightning cables for iPhones (ID: B0BPLX388R), USB C to Lightning cables (ID: B0CFG7V2QX), USB A to Type C cables (ID: B0C7Q3X76Q), and USB 2.0 extension cables (ID: B0BV276QMH).


In [56]:
def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient("http://localhost:6333")

    retrieved_context = reteriver_data(
        question,
        qdrant_client,
        top_k
    )

    processed_context = process_context(
        retrieved_context[0],
        retrieved_context[1],
        retrieved_context[2],
        retrieved_context[3]
    )

    prompt = build_prompt(
        processed_context,
        question
    )

    answer = generate_chat(prompt)

    return answer

In [61]:
res=rag_pipeline("what type of cords can i get",5)

In [62]:
res

'You can get Lightning cables for iPhones (ID: B0BPLX388R), USB A to Type C cables (ID: B0C7Q3X76Q), and short USB 2.0 extension cables (ID: B0BV276QMH).'